# FleetLogix - Avance 4: Arquitectura Cloud AWS

**Entrega de diseño, sin despliegue.** Región propuesta: `us-east-2`. Flujo: **app -> API Gateway -> Lambda -> DynamoDB**. EventBridge distribuye eventos a ETA, desvíos y archivo S3; CloudWatch supervisa. RDS PostgreSQL corresponde a la migración planeada del origen relacional.

## Decisiones principales

- **DynamoDB**: estado actual de una entrega; **S3**: histórico JSON por día; **RDS**: datos relacionales migrados.
- **Cuatro Lambda**: ingreso/entrega completada, ETA, desvío y archivo.
- **Seguridad**: JWT, IAM mínimo, cifrado, backups RDS 7 días, RDS privada.
- **Costo**: estimación teórica de **USD 23.10/mes**, sin Free Tier; no es cotización.
- **No desplegado**: la demo siguiente es completamente local.

In [1]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
if not (root / "Scripts" / "Avance_4").exists():
    root = root.parent
sys.path.insert(0, str(root / "Scripts" / "Avance_4"))
os.environ["LOCAL_DEMO"] = "1"
from lambda_functions.logic import check_delivery_completed, estimate_eta, detect_route_deviation
print("Cuatro Lambdas documentadas; modo local activo")

Cuatro Lambdas documentadas; modo local activo


In [2]:
ejemplo = {"event_id":"e-101", "delivery_id":"101", "event_type":"delivery_status", "occurred_at":"2026-11-18T14:20:00Z", "status":"delivered"}
assert check_delivery_completed(ejemplo)
eta = estimate_eta({"occurred_at":"2026-11-18T14:00:00Z", "remaining_distance_km":10, "average_speed_kmh":40})
desvio = detect_route_deviation({"current_position":{"lat":4.75,"lon":-74.07},"expected_waypoint":{"lat":4.71,"lon":-74.07}})
print("Entrega completada:", True, "| ETA min:", eta["estimated_minutes"], "| Alerta:", desvio["alert"])

Entrega completada: True | ETA min: 15.0 | Alerta: True


In [3]:
import subprocess
resultado = subprocess.run([sys.executable, str(root / "Scripts" / "Avance_4" / "test_lambda_logic.py")], capture_output=True, text=True, check=True)
print(resultado.stdout)

PRUEBA LOCAL AWS COMPLETA: OK
API -> DynamoDB simulado: estado final = delivered
ETA simple: 15.0 minutos | alerta de desvío: SI
Eventos archivados en S3 simulado: 2
Evento repetido rechazado sin duplicación: OK
0 conexiones AWS | 0 costos | 0 cambios a PostgreSQL/Snowflake



## Interpretación y límites

La actualización de una entrega se demuestra **localmente** (estado consultable inmediatamente). ETA = distancia/velocidad y desvío = distancia a waypoint, no rutas reales. El PDF del paquete contiene el diagrama, el plan de migración, métricas de CloudWatch, seguridad IAM y la tabla de costos. No afirmamos haber migrado PostgreSQL a RDS ni desplegado API Gateway.